# Домашнее задание: эмбеддинги на уровне слов и поисковая система запросов

В общих чертах, ваша задача будет заключаться в следующем:
- Обучить модель fasttext на романе Льва Толстого "Война и мир"
- Дообучить (продолжить обучение) полученные эмбеддинги на вопросах из Quora, чтобы учесть новые токены и постоянно меняющийся семантический смысл
- Получить эмбеддинги для существующих вопросов Quora и сохранить их для дальнейшего использования
- Реализовать способ поиска наиболее близкого совпадения среди вопросов Quora для нового текста

Удачи!

# Обучение начальных эмбеддингов (4 балла)

Вашей первой задачей будет обучение эмбеддингов на романе Льва Толстого "Война и мир" с использованием расширения word2vec под названием fasttext.

Есть определенные отличия от стандартного word2vec, о которых можно прочитать, например, [здесь](https://github.com/facebookresearch/fastText?tab=readme-ov-file), но для всех практических целей проще всего воспринимать fasttext как word2vec на стероидах.

Пожалуй, самым важным отличием является то, что словарь модели строится не только с использованием токенов на уровне слов, но и их n-грамм, например, для слова "peace" полученные токены при использовании 2-грамм будут выглядеть как (peace, pe, ea, ac, ce).

Это позволяет модели не только обрабатывать слова вне словаря путем объединения соответствующих подтокенов, но и в некоторых случаях избавляет от необходимости нормализации слов (стемминга, лемматизации), сохраняя семантику в морфологически богатых языках.

In [ ]:
!pip install unidecode
!pip install faiss-cpu
!pip install gensim unidecode

import re
from typing import Union, List, Tuple, Callable

import nltk
import faiss
import numpy as np
from unidecode import unidecode
from gensim.models import FastText
from gensim.models.word2vec import PathLineSentences

nltk.download('punkt_tab')
seed = 42

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\savro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Загрузите текстовый файл "Война и мир" из репозитория проекта Gutenberg.

In [2]:
!wget "https://www.gutenberg.org/files/2600/2600-0.txt" -O war_and_peace_raw.txt

--2026-03-19 22:40:58--  https://www.gutenberg.org/files/2600/2600-0.txt
Loaded CA certificate '/usr/ssl/certs/ca-bundle.crt'
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3359405 (3.2M) [text/plain]
Saving to: 'war_and_peace_raw.txt'

     0K .......... .......... .......... .......... ..........  1%  233K 14s
    50K .......... .......... .......... .......... ..........  3%  431K 11s
   100K .......... .......... .......... .......... ..........  4%  688K 8s
   150K .......... .......... .......... .......... ..........  6% 59.2K 19s
   200K .......... .......... .......... .......... ..........  7%  767K 16s
   250K .......... .......... .......... .......... ..........  9% 1003K 14s
   300K .......... .......... .......... .......... .......... 10% 2.93M 12s
   350K .......... .......... ....

In [3]:
wap_raw_file_path = 'war_and_peace_raw.txt'
wap_cleaned_file_path = 'war_and_peace_cleaned.txt'

Обратите внимание, что текст содержит информацию, избыточную для задачи, поэтому сначала нам нужно выполнить предобработку файла.

In [4]:
!head war_and_peace_raw.txt

п»їThe Project Gutenberg eBook of War and Peace, by Leo Tolstoy

This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online at
www.gutenberg.org. If you are not located in the United States, you
will have to check the laws of the country where you are located before
using this eBook.



In [ ]:
with open(wap_raw_file_path, 'r', encoding='utf-8') as file:
    text = file.read()

start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK WAR AND PEACE ***"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK WAR AND PEACE ***"
text = text.split(start_marker, 1)[-1]
text = text.split(end_marker, 1)[0]

lines = text.split("\n")
filtered_lines = []
for line in lines:
    line = line.strip()
    if not line or line.startswith("CHAPTER") or line.isupper():
        continue
    filtered_lines.append(line)

cleaned_text = "\n".join(filtered_lines[2:])

По историческим причинам текст содержит некоторые символы, не относящиеся к ASCII. Один из способов справиться с этим — привести их к ближайшему эквиваленту в ASCII.

Обратите внимание, что это не всегда обязательно, и в некоторых случаях может негативно сказаться на производительности модели.

*Короче говоря, если вы считаете, что эти символы нужны, оставьте их.*

In [ ]:
cleaned_text_ascii = unidecode(cleaned_text)

Еще одна важная вещь — понимать свои данные и то, как модель обучает свои веса.

В нашем случае word2vec (fasttext) обрабатывает текст построчно, проходя по строке с заданным окном для представления контекста. Однако текст "Войны и мира" от Gutenberg разбит на строки несколько произвольно, так что слова в конкретной строке не обязательно принадлежат одному и тому же предложению, и поэтому окно контекста будет ошибочно считать их контекстно-зависимыми.

Один из способов это исправить — сначала разбить текст на предложения. К счастью для нас, весь текст "Войны и мира" помещается в оперативную память.

In [ ]:
sentences = nltk.sent_tokenize(cleaned_text_ascii)

sentences[:10]

['"Well, Prince, so Genoa and Lucca are now just family estates of the\nBuonapartes.',
 "But I warn you, if you don't tell me that this means war,\nif you still try to defend the infamies and horrors perpetrated by that\nAntichrist--I really believe he is Antichrist--I will have nothing\nmore to do with you and you are no longer my friend, no longer my\n'faithful slave,' as you call yourself!",
 'But how do you do?',
 'I see I\nhave frightened you--sit down and tell me all the news."',
 'It was in July, 1805, and the speaker was the well-known Anna Pavlovna\nScherer, maid of honor and favorite of the Empress Marya Fedorovna.',
 'With these words she greeted Prince Vasili Kuragin, a man of high\nrank and importance, who was the first to arrive at her reception.',
 'Anna\nPavlovna had had a cough for some days.',
 'She was, as she said, suffering\nfrom la grippe; grippe being then a new word in St. Petersburg, used\nonly by the elite.',
 'All her invitations without exception, written in

Как видите, в каждом предложении есть много нежелательных символов: произвольные переносы строк (\n), двойные дефисы (--) и т.д.

Для нашей задачи (обучение эмбеддингов fasttext) мы можем захотеть предварительно обработать текст, то есть избавиться от пунктуации и лишних символов, перевести каждое слово в нижний регистр. Мы также могли бы применить стемминг или лемматизацию, но fasttext дает нам свободу этого не делать, сохраняя некоторую информацию, связанную с морфологией.

In [ ]:
def clean_wap_sentence(sentence: str) -> str:
    """
    Очищает предложение, выполняя следующие операции:
    1. Заменяет символы переноса строки (`\n`) пробелом.
    2. Заменяет двойные дефисы (`--`) пробелом.
    3. Удаляет всю пунктуацию, кроме апострофов.
    4. Переводит предложение в нижний регистр.

    Аргументы:
        sentence (str): Входное предложение для очистки.

    Возвращает:
        str: Очищенное предложение.
    """
    sentence = sentence.replace('\n', ' ')
    sentence = sentence.replace('--', ' ')
    sentence = re.sub(r"[^\w\d\s']+", '', sentence)
    return sentence.lower().strip()

In [ ]:
processed_sentences = []

for sentence in sentences:
    cleaned = clean_wap_sentence(sentence)
    if cleaned:
        processed_sentences.append(cleaned)

final_text = "\n".join(processed_sentences)

with open(wap_cleaned_file_path, 'w') as outfile:
    outfile.write(final_text)

wap_sentences_count = len(processed_sentences)

In [ ]:
!sed -n 1,4p war_and_peace_cleaned.txt

well prince so genoa and lucca are now just family estates of the buonapartes
but i warn you if you don't tell me that this means war if you still try to defend the infamies and horrors perpetrated by that antichrist i really believe he is antichrist i will have nothing more to do with you and you are no longer my friend no longer my 'faithful slave' as you call yourself
but how do you do
i see i have frightened you sit down and tell me all the news


Теперь, когда у нас есть предобработанные данные, давайте, наконец, обучим нашу модель!

Для этой задачи мы будем использовать [реализацию Fasttext от Gensim](https://radimrehurek.com/gensim/models/fasttext.html). Она имеет небольшие накладные расходы по сравнению с [базовой реализацией](https://fasttext.cc/), но предлагает очень удобный API, который также поддерживает дообучение существующих моделей.


**Пожалуйста, внимательно прочитайте эти примечания, чтобы пройти тесты:**

0. Рекомендуется установить learning rate (скорость обучения) на 3e-2. Fasttext в Gensim будет самостоятельно линейно уменьшать это значение с каждой эпохой.
1. Вы можете экспериментировать с размером вектора,
но знайте, что это сильно повлияет на производительность.
Значения ниже 64 могут привести к тому, что эмбеддинги будут с трудом корректно представлять слова.
Значения выше 300 замедлят работу и, скорее всего, не принесут никакой компенсации за это для текущего объема данных.
Мы рекомендуем значение 100 как золотую середину, чтобы безопасно пройти тест.
2. Рекомендуется использовать режим skip-gram. Он обычно дает лучшее семантическое представление изученных векторов ценой относительно медленного обучения, что для нас некритично.
3. 5 эпох, вероятно, достаточно для имеющегося у вас объема данных, но можете поэкспериментировать.
4. Не забудьте установить seed, хотя это, вероятно, не сделает вашу модель полностью детерминированной, если вы используете более одного потока.
5. Размер окна — важный параметр. Чем шире окно, тем более широкий контекст на уровне предложения захватывает данный эмбеддинг за счет собственной семантической уникальности. Можете поэкспериментировать. Размер окна 5, вероятно, будет хорошей отправной точкой для безопасного прохождения блока проверок (assert wall).

Обратите внимание, что мы не используем разбиение на train\test. Вы, человек, являетесь главным судьей в этой задаче. Если вы считаете, что эмбеддинги имеют смысл (и они проходят тесты), пусть так и будет.

In [ ]:
wap_sentences = PathLineSentences(wap_cleaned_file_path)

model = FastText(
    sentences=wap_sentences,
    vector_size=100,
    window=5,
    sg=1,
    alpha=0.03,
    min_count=5,
    seed=seed,
    epochs=5,
    workers=1
)

model.save("wap_fasttext_model.bin")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Ниже вы можете поэкспериментировать с обученной моделью.

In [12]:
print(*model.wv.most_similar('peace', topn=10), sep='\n')

('hope', 0.8743038177490234)
('religion', 0.861891508102417)
('conquer', 0.8520426750183105)
('problem', 0.8507330417633057)
('freedom', 0.8449881672859192)
('disgrace', 0.8448716998100281)
('faith', 0.8441885709762573)
('selfsacrifice', 0.8427587747573853)
('secrecy', 0.8400357365608215)
('perish', 0.8396977782249451)


Fasttext позволяет нам восстанавливать слова вне словаря, используя n-граммы слов...

In [13]:
'computation' in model.wv.key_to_index

False

...даже если они могут быть слишком далеки от изученного контекста.

In [14]:
print(*model.wv.most_similar('computation', topn=10), sep='\n')

('deputation', 0.9727035164833069)
('gravitation', 0.9685028791427612)
('subordination', 0.9664888381958008)
('determination', 0.9660364389419556)
('combination', 0.9660340547561646)
('consultation', 0.965701162815094)
('observation', 0.9656822085380554)
('participation', 0.9647431969642639)
('manifestation', 0.9639513492584229)
('reputation', 0.9621769189834595)


In [ ]:
model_vocabulary = set(model.wv.key_to_index.keys())
most_similar_to_peace = list(zip(*model.wv.most_similar('peace', topn=50)))[0]

assert model.vector_size >= 64 and model.vector_size <= 300, 'Пожалуйста, проверьте размер ваших эмбеддингов.'
assert model.sg == 1, 'Пожалуйста, используйте метод skip-gram для согласованности. Кроме того, несмотря на более высокую скорость, CBOW обычно генерирует худшие эмбеддинги для редких слов.'
assert model.alpha == 3e-2, 'Ожидается, что вы переопределите дефолтное значение alpha для этой задачи.'
assert len(model_vocabulary) > 5500 and len(model_vocabulary) < 7500, 'Что-то не так с вашей токенизацией. Проверьте пайплайн и используйте стандартный *ngram=1*'
assert 'religion' in most_similar_to_peace, 'Эмбеддинги выглядят странно. Убедитесь, что вы следуете инструкциям!'

print('Congrats!')

Congrats!


# Дообучение на датасете Quora (3 балла)

Теперь, когда у нас есть эмбеддинги, представляющие семантическое пространство в том виде, в каком его воспринимал бы англоязычный Лев Толстой, давайте перейдем к более современным представлениям.

Одна из проблем заключается в том, что словарю нашей модели не хватает знаний о современных словах. И хотя мы можем восстановить такие слова с помощью n-грамм fasttext, им не хватает контекстного значения.
Другая проблема заключается в том, что контекст использования определенных слов мог измениться за прошедшие годы.

Следовательно, наша задача будет заключаться не в перезаписи изученных векторов, а в их обогащении свежими данными.

In [ ]:
!wget https://www.dropbox.com/s/obaitrix9jyu84r/quora.txt?dl=1 -O ./quora_raw.txt

--2026-03-19 22:41:54--  https://www.dropbox.com/s/obaitrix9jyu84r/quora.txt?dl=1
Loaded CA certificate '/usr/ssl/certs/ca-bundle.crt'
Resolving www.dropbox.com (www.dropbox.com)... 162.125.70.18, 2620:100:6026:18::a27d:4612
Connecting to www.dropbox.com (www.dropbox.com)|162.125.70.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/p0t2dw6oqs6oxpd6zz534/quora.txt?rlkey=bjupppwua4zmd4elz8octecy9&dl=1 [following]
--2026-03-19 22:41:55--  https://www.dropbox.com/scl/fi/p0t2dw6oqs6oxpd6zz534/quora.txt?rlkey=bjupppwua4zmd4elz8octecy9&dl=1
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://ucd5b1e56acb42bdfec2794439a7.dl.dropboxusercontent.com/cd/0/inline/C887gROo2E7DPvqZCiihQSje_nP6BlZzkJw1k96Rj5jWfWVded3JZpjBk-LT2BG35OwMoq9EgHkn8tVoMy3slOdYSV7om69UZpoy7SPsf-ygL_vy7drRqHU8HENsv1wIs0A/file?dl=1# [following]
--2026-03-19 22:41:56--  https://ucd5b1e56acb42bdfec279

In [ ]:
!sed -n 1,5p quora_raw.txt

Can I get back with my ex even though she is pregnant with another guy's baby?
What are some ways to overcome a fast food addiction?
Who were the great Chinese soldiers and leaders who fought in WW2?
What are ZIP codes in the Bay Area?
Why was George RR Martin critical of JK Rowling after losing the Hugo award?


Как и раньше, нам нужно стандартизировать данные для обучения.

К счастью для нас, каждая строка уже состоит из полного предложения (что еще лучше, в ASCII!).

In [ ]:
def preprocess_line(line: str) -> str:
    """
    Предобрабатывает одну строку текста путем:
    1. Удаления всей пунктуации, кроме апострофов.
    2. Перевода текста в нижний регистр.

    Аргументы:
        line (str): Входная строка текста для предобработки.

    Возвращает:
        str: Предобработанная строка текста.
    """

    line = re.sub(r"[^\w\d\s']+", '', line)
    return line.lower().strip()


def preprocess_file(input_file_path: str, output_file_path: str) -> int:
    """
    Читает текстовый файл построчно, предобрабатывает каждую строку
    и записывает обработанные строки в новый файл.

    Аргументы:
        input_file_path (str): Путь к входному текстовому файлу.
        output_file_path (str): Путь к выходному текстовому файлу, где будут сохранены обработанные строки.

    Возвращает:
        int: Общее количество строк, обработанных во входном файле.
    """

    lines_processed = 0

    with open(input_file_path, 'r', encoding='utf-8') as infile, \
         open(output_file_path, 'w', encoding='utf-8') as outfile:
        for line in infile:
            cleaned_line = preprocess_line(line)
            outfile.write(cleaned_line + '\n')
            lines_processed += 1
            
    return lines_processed

quora_raw_file_path = 'quora_raw.txt'
quora_processed_file_path = 'quora_processed.txt'
quora_sentences_count = preprocess_file(quora_raw_file_path, quora_processed_file_path)

print(f'Обработано строк: {quora_sentences_count}')


Обработано строк: 537272


Загвоздка здесь в том, что мы продолжаем обучать модель на более крупном датасете с совершенно другой тематикой. Поэтому, чтобы не полностью перезаписать существующую семантику "Войны и мира", было бы неплохо продолжить с того места, где мы остановились (в плане скорости обучения) и/или не слишком увлекаться количеством эпох.  

In [ ]:
quora_sentences = PathLineSentences(quora_processed_file_path)
finetuned_model = FastText.load("wap_fasttext_model.bin")
finetuned_model.build_vocab(corpus_iterable=quora_sentences, update=True)

finetuned_model.train(
    corpus_iterable=quora_sentences,
    total_examples=finetuned_model.corpus_count,
    epochs=5, 
    start_alpha=0.0001
)

finetuned_model.save("wap_quora_fasttext_model.bin")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [27]:
finetuned_model_vocabulary = set(finetuned_model.wv.key_to_index.keys())
new_words = finetuned_model_vocabulary - model_vocabulary

print("Количество добавленных новых слов:", len(new_words))
print("Новые слова:", list(new_words)[10:20])

Количество добавленных новых слов: 23878
Новые слова: ['milan', 'departed', 'tort', 'vevo', 'versus', 'concert', 'tgc', 'grades', 'coordinate', 'usmle']


In [ ]:
def print_comparison(
    model1: Union[FastText, None],
    model2: Union[FastText, None],
    word: str = 'peace',
    top_n: int = 10
) -> None:
    """
    Вспомогательная функция, которая сравнивает топ-N наиболее похожих слов для заданного слова-запроса в двух моделях FastText.
    Выводит сравнение в табличном формате для визуальной оценки изменений после дообучения.

    Аргументы:
        model1 (FastText | None): Первая модель FastText (оригинальная или до дообучения).
        model2 (FastText | None): Вторая модель FastText (после дообучения).
        word (str, optional): Слово, для которого извлекаются похожие слова.
        top_n (int, optional): Количество извлекаемых похожих слов для сравнения.

    Возвращает:
        None: Выводит результаты напрямую в консоль.
    """
    q0 = model1.wv.most_similar(word, topn=top_n)
    q1 = model2.wv.most_similar(word, topn=top_n)

    print(f'Запрос: {word}\n')
    print(f"{'pos':<5} {'model_1':<15} {'score_1':<10} {'model_2':<15} {'score_2':<10}")
    print("-" * (5 + 15 + 10 + 15 + 10))

    for pos, (word0, score0), (word1, score1) in zip(range(top_n), q0, q1):
        print(f"{pos:<5} {word0:<15} {score0:.4f}     {word1:<15} {score1:.4f}")


In [ ]:
print_comparison(model, finetuned_model, word='war', top_n=10)

Запрос: war

pos   model_1         score_1    model_2         score_2   
-------------------------------------------------------
0     revolution      0.8161     warg            0.9357
1     communication   0.8107     abolition       0.9225
2     disposition     0.8099     revolution      0.9224
3     campaigns       0.8083     macroevolution  0.9222
4     vital           0.8053     microevolution  0.9207
5     campaign        0.8047     evolution       0.9190
6     dissolution     0.7980     actuarial       0.9167
7     trivial         0.7967     communication   0.9162
8     trial           0.7967     devolution      0.9153
9     description     0.7964     disposition     0.9124


In [ ]:
assert finetuned_model.alpha <= model.min_alpha, 'установка слишком высокой скорости обучения приведет к тому, что модель забудет предыдущую информацию, особенно при дообучении на более крупном датасете!'
assert len(new_words) > 20_000 and len(new_words) < 30_000, 'Скорее всего, что-то не так с вашим пайплайном предобработки. После дообучения должно быть больше новых слов!'
assert 'religion' in list(zip(*finetuned_model.wv.most_similar('peace', topn=50)))[0], 'Эмбеддинги кардинально изменились! Подумайте об установке меньшего start_alpha и меньшего количества эпох.'
assert 'evolution' in list(zip(*finetuned_model.wv.most_similar('war', topn=50)))[0], 'Эмбеддинги не изменились так, как ожидалось! Подумайте об установке более высокого start_alpha и большего количества эпох, проверьте параметр *total_examples*.'

print('Gratz!')

Gratz!


На практике вы можете поэкспериментировать с различными стратегиями дообучения, например, варьируя параметр alpha, обучая большее количество эпох или даже пытаясь "заморозить" некоторые эмбеддинги, обновляя только определенные части словаря.

# Реализуйте способ поиска похожих вопросов Quora для нового запроса (3 балла)

## Простой поиск на основе массивов numpy

Далее, если мы собираемся создать систему семантического поиска, необходимо уметь сопоставлять новое предложение с единым вектором. Самый простой способ сделать это — усреднить индивидуальные эмбеддинги на уровне слов.

В качестве альтернативы вы можете попробовать стратегии пулинга (min, max, mean, softmax) и [TF-IDF](https://www.geeksforgeeks.org/understanding-tf-idf-term-frequency-inverse-document-frequency/) . Подход TF-IDF требует предварительного вычисления значений IDF по всему существующему корпусу и умножения их на TF для конкретного слова в заданном предложении.

Кроме того, ожидается, что позже вы будете использовать косинусное сходство в качестве метрики векторного расстояния, поэтому было бы хорошей идеей нормализовать все векторы перед помещением их в базу данных, так как это фактически превратит вычисление косинусного сходства во взятие скалярного произведения, что ускорит работу:
$$
cos(\phi) = \frac{<u, v>}{|u||v|}.
$$

In [ ]:
def normalize_vector(embedding: np.ndarray) -> np.ndarray:
    """
    Нормализует заданный вектор до единичной длины.

    Аргументы:
        embedding (np.ndarray): Массив NumPy, представляющий вектор для нормализации.

    Возвращает:
        np.ndarray: Нормализованный вектор единичной длины.
    """

    norm = np.linalg.norm(embedding)
    
    if norm == 0:
        return embedding
        
    return embedding / norm


def get_text_embedding(text: str, model: FastText) -> np.ndarray:
    """
    Вычисляет эмбеддинг для заданного текста, используя предварительно обученную модель FastText.
    Эмбеддинг — это среднее значение эмбеддингов слов для слов в тексте, нормализованное до единичной длины.

    Аргументы:
        text (str): Входной текст для преобразования в вектор.
        model (FastText): Модель FastText для генерации эмбеддингов.

    Возвращает:
        np.ndarray: Нормализованный эмбеддинг для входного текста.
                    Если в тексте не найдено допустимых слов, возвращает нулевой вектор.
    """
    cleaned_text = preprocess_line(text) 
    
    words = cleaned_text.split()
    if not words:
        return np.zeros(model.vector_size)
    
    word_vectors = model.wv[words]
    avg_embedding = np.mean(word_vectors, axis=0)
    
    return normalize_vector(avg_embedding)

In [ ]:
question = "Love peace"
text_embedding = get_text_embedding(question, finetuned_model)
dummy_word_embedding = (finetuned_model.wv['love'] + finetuned_model.wv['peace']) / 2

assert text_embedding.shape[0] == finetuned_model.wv.vector_size, "Проверьте ось, вдоль которой агрегируются значения"
assert all(normalize_vector(dummy_word_embedding) == text_embedding), "Пока вам нужно вернуть нормализованный усредненный эмбеддинг, ничего сложного!"

print('Good job!')

Good job!


Наконец, давайте реализуем поиск похожих предложений.

Ваша задача будет заключаться в сопоставлении каждому вопросу Quora эмбеддинга и хранении этих векторов таким образом, чтобы можно было легко проводить операцию поиска.

Чтобы это было похоже на поисковую систему, новый пользовательский ввод также должен преобразовываться в эмбеддинг для выполнения, собственно, поиска в нашем существующем хранилище эмбеддингов.

Сначала давайте сделаем это простым способом: создадим массив numpy, в котором будут храниться эмбеддинги существующих вопросов Quora.

Обязательно примените ту же предобработку, которую мы использовали во время обучения модели!

In [ ]:
def create_embeddings_storage(quora_file: str, model: FastText) -> np.ndarray:
    """
    Создает массив NumPy с нормализованными эмбеддингами для всех вопросов в датасете.

    Каждая строка во входном файле представляет вопрос, который предварительно обрабатывается с помощью
    функции 'preprocess_line', а его эмбеддинг получается с помощью функции 'get_text_embedding'.

    Аргументы:
        quora_file (str): Путь к файлу, содержащему предобработанные вопросы, по одному на строку.
        model (FastText): Модель FastText, используемая для генерации эмбеддингов.

    Возвращает:
        np.ndarray: 2D-массив NumPy, где каждая строка соответствует нормализованному эмбеддингу
                    вопроса в датасете. Размерность: (num_questions, embedding_dim).
    """
    embeddings_list: List[np.ndarray] = []
    
    with open(quora_file, 'r', encoding='utf-8') as f:
        for line in f:
            embedding = get_text_embedding(line.strip(), model)
            embeddings_list.append(embedding)

    return np.array(embeddings_list)

In [ ]:
embeddings_storage_np = create_embeddings_storage("quora_processed.txt", finetuned_model)

In [45]:
with open(quora_processed_file_path, 'r') as f:
    for line in f:
        test_embedding = get_text_embedding(line, finetuned_model)
        break

assert embeddings_storage_np.shape == (quora_sentences_count, finetuned_model.vector_size), "Хранилище векторов должно иметь размер (num_quora_sentences, embedding_dim)"
assert embeddings_storage_np[0].mean() == test_embedding.mean(), 'Эмбеддинг первого вопроса quora не соответствует первой записи в хранилище!'
print('Пока все выглядит отлично!')

Пока все выглядит отлично!


In [ ]:
def find_closest_match_np(query: str, model, embeddings_storage: np.ndarray, k: int = 1) -> Tuple[List[int], List[float]]:
    """
    Находит наиболее близкие совпадения для нового вопроса в базе данных с использованием косинусного сходства.

    Функция предобрабатывает входной вопрос, вычисляет его вектор, а затем считает
    косинусное сходство между вектором нового вопроса и существующей базой эмбеддингов.
    Она возвращает индексы топ-k наиболее похожих вопросов вместе с их оценками сходства.

    Параметры:
        query (str): Входной вопрос для сравнения.
        model: Обученная модель FastText для генерации эмбеддингов.
        embeddings_storage (np.ndarray): Массив numpy, содержащий предварительно вычисленные и нормализованные векторы всех вопросов в базе.
        k (int, optional): Количество ближайших совпадений для возврата. По умолчанию 1.

    Возвращает:
        Tuple[List[int], List[float]]:
            - List[int]: Список индексов топ-k наиболее похожих вопросов в базе данных.
            - List[float]: Список оценок сходства, соответствующих этим совпадениям.
    """

    cleaned_query = preprocess_line(query)
    query_embedding = get_text_embedding(cleaned_query, model)
    similarities = np.dot(embeddings_storage, query_embedding)

    top_k_indices = np.argsort(similarities)[-k:][::-1].tolist()
    top_k_similarities = similarities[top_k_indices].tolist()

    return top_k_indices, top_k_similarities

Пока мы будем хранить все вопросы Quora в оперативной памяти (RAM) с использованием списка Python.

На практике вы бы рассмотрели отдельную базу данных с извлечением по индексу, выполняемым за константное время.

In [ ]:
quora_questions_list = []
with open(quora_raw_file_path, 'r', encoding='utf-8') as file:
    for line in file:
        quora_questions_list.append(line.strip())

print(f"Всего обработано вопросов: {len(quora_questions_list)}")

Всего обработано вопросов: 537272


In [ ]:
def fetch_and_display_closest_match(query_function: Callable[..., Tuple[List[int], List[float]]], **kwargs) -> Tuple[List[int], List[float]]:
    """
    Вспомогательная функция для извлечения и отображения наиболее подходящих вопросов
    из датасета Quora для заданного запроса.

    Функция принимает запрос, использует переданную функцию поиска для извлечения
    топ-k наиболее похожих вопросов и выводит их вместе с оценками сходства.

    Аргументы:
        query_function (Callable): Функция, принимающая параметры запроса и возвращающая
                                    индексы топ-k наиболее похожих вопросов и их оценки сходства.
                                    Она должна возвращать кортеж (List[int], List[float]).
        **kwargs: Дополнительные аргументы для передачи в функцию запроса, включая текст запроса.

    Возвращает:
        Tuple[List[int], List[float]]: Кортеж, содержащий:
            - Список индексов, соответствующих топ-k наиболее похожим вопросам в датасете.
            - Список оценок сходства для каждого из этих вопросов.
    """

    top_k_indices, top_k_similarities = query_function(**kwargs)
    top_k_questions = [quora_questions_list[i] for i in top_k_indices]
    
    print(f"Запрос: {kwargs['query']}")
    print("\nЛучшие совпадения:")
    for i, (question, similarity) in enumerate(zip(top_k_questions, top_k_similarities), 1):
        print(f"{i}. {question} (Сходство: {similarity:.4f})")

    return top_k_indices, top_k_similarities


In [ ]:
new_question = "How can I find inner peace?"

top_k_indices, top_k_similarities = fetch_and_display_closest_match(query_function=find_closest_match_np,
                                                                    query=new_question, model=finetuned_model,
                                                                    embeddings_storage=embeddings_storage_np,
                                                                    k=10)

assert 446084 in top_k_indices, 'Ваши эмбеддинги выглядят странно. = ('

Запрос: How can I find inner peace?

Лучшие совпадения:
1. How can I create inner peace? (Сходство: 0.9945)
2. How can I find passion? (Сходство: 0.9936)
3. How can I find inspiration? (Сходство: 0.9930)
4. How can I find a career mentor? (Сходство: 0.9928)
5. How can I find a successful mentor? (Сходство: 0.9926)
6. How can I find a programmer mentor? (Сходство: 0.9921)
7. How can I find someone's MSN profile? (Сходство: 0.9918)
8. How can I truly find happiness? (Сходство: 0.9911)
9. How can I find a startup mentor? (Сходство: 0.9911)
10. How can I prevent business failure? (Сходство: 0.9911)


Пожалуйста, не стесняйтесь экспериментировать с вашей новой поисковой системой!

In [51]:
new_question = "What is the future of artificial intelligence?"

_, _ = fetch_and_display_closest_match(query_function=find_closest_match_np,
                                       query=new_question, model=finetuned_model,
                                       embeddings_storage=embeddings_storage_np,
                                       k=10)


Запрос: What is the future of artificial intelligence?

Лучшие совпадения:
1. What is the future of artificial intelligence? (Сходство: 1.0000)
2. What is the future of Social Media? (Сходство: 0.9982)
3. What is the future of Pharmaceutical industry? (Сходство: 0.9981)
4. What is the scope of Artificial Intelligence? (Сходство: 0.9981)
5. What is the future of Chinese economy? (Сходство: 0.9979)
6. What is the nature of public administration? (Сходство: 0.9979)
7. What is the indefinite integral of [math]\sin (x^3)[/math] ? (Сходство: 0.9977)
8. What is the future of electrical grid? (Сходство: 0.9976)
9. What is the future of social networking? (Сходство: 0.9975)
10. What is the future of Indonesia's economy? (Сходство: 0.9975)


In [52]:
import time

iterations = 100

start_time = time.time()

for _ in range(iterations):
    _, _ = find_closest_match_np('hello there', finetuned_model, embeddings_storage_np, k=10)

time_elapsed = time.time() - start_time
average_time = time_elapsed / iterations

print(f'Прошло времени: {time_elapsed:.4f} секунд.')
print(f'Среднее время: {average_time:.4f} секунд.')

Прошло времени: 6.8166 секунд.
Среднее время: 0.0682 секунд.


Обратите внимание, что на практике частые запросы кэшируются.

# Поиск в векторной базе данных

Однако главная проблема заключается в том, что часто ваши данные будут слишком велики, чтобы поместиться в RAM, не говоря уже о VRAM, поэтому нужен вариант лучше, чем поиск по массиву numpy. Кроме того, мы не можем позволить себе тратить O(n) времени на каждый поисковый запрос.

Векторные базы данных, то есть базы данных, специально разработанные для хранения векторов и проведения быстрых операций извлечения, решают проблему скорости за счет необходимости хранить данные в RAM. На практике коллекция машин (шардов) может размещать различные фрагменты данных во взаимосвязанной форме.

Ниже представлен уже реализованный способ создания хранилища эмбеддингов.

Вы можете прочитать больше о библиотеке [FAISS](https://faiss.ai/index.html) и методе[HNSW](https://arxiv.org/abs/1603.09320), который, вкратце, представляет собой поиск k-ближайших соседей на стероидах.

In [ ]:
def build_faiss_hnsw_index(dimension: int, ef_construction: int = 200, M: int = 32) -> faiss.IndexHNSWFlat:
    """
    Создает индекс FAISS HNSW для вычисления косинусного сходства.

    Эта функция инициализирует индекс HNSW (Hierarchical Navigable Small World) для эффективного приближенного поиска ближайших соседей
    на основе косинусного сходства, используя нормализованные эмбеддинги модели FastText.

    Параметры:
        dimension (int): Размерность эмбеддингов (размер каждого вектора).
        ef_construction (int, optional): Параметр компромисса между скоростью построения индекса и точностью. По умолчанию 200.
        M (int, optional): Количество соседей в графе, контролирующее баланс памяти и точности. По умолчанию 32.

    Возвращает:
        index (faiss.IndexHNSWFlat): Инициализированный индекс FAISS HNSW.
    """
    index = faiss.IndexHNSWFlat(dimension, M)
    index.hnsw.efConstruction = ef_construction
    index.metric_type = faiss.METRIC_INNER_PRODUCT
    return index


def populate_faiss_index(index: faiss.Index, model, dataset_path: str, batch_size: int = 10000):
    """
    Заполняет индекс FAISS HNSW нормализованными эмбеддингами из датасета.

    Функция читает датасет построчно, предобрабатывает каждый вопрос, вычисляет его эмбеддинг
    и добавляет эмбеддинги в индекс FAISS пакетами (батчами).

    Параметры:
        index (faiss.Index): Индекс FAISS для заполнения.
        model: Обученная модель FastText для генерации эмбеддингов.
        dataset_path (str): Путь к файлу датасета (один вопрос на строку).
        batch_size (int, optional): Количество вопросов для одновременной обработки. По умолчанию 10000.
    """
    with open(dataset_path, "r", encoding='utf-8') as f:
        buffer = []
        for line in f:
            question = preprocess_line(line.strip())
            embedding = get_text_embedding(question, model)
            buffer.append(embedding)

            if len(buffer) >= batch_size:
                index.add(np.array(buffer, dtype=np.float32))
                buffer = []

        if buffer:
            index.add(np.array(buffer, dtype=np.float32))


def search_faiss_index(embeddings_storage: faiss.Index, query: str, model, k: int = 5) -> Tuple[List[int], List[float]]:
    """
    Выполняет поиск в индексе FAISS для нахождения наиболее близких совпадений с запросом.

    Функция вычисляет эмбеддинг для входного запроса, ищет в индексе FAISS наиболее похожие вопросы
    и возвращает индексы и расстояния (косинусное сходство) топ-k совпадений.

    Параметры:
        embeddings_storage (faiss.Index): Индекс FAISS для поиска.
        query (str): Входная строка запроса для поиска.
        model: Обученная модель FastText для генерации эмбеддингов.
        k (int, optional): Количество возвращаемых ближайших совпадений. По умолчанию 5.

    Возвращает:
        Tuple[List[int], List[float]]:
            - List[int]: Список индексов топ-k наиболее похожих вопросов.
            - List[float]: Список расстояний (косинусного сходства) топ-k результатов.
    """
    query_embedding = get_text_embedding(query, model)

    top_k_distances, top_k_indices = embeddings_storage.search(np.array([query_embedding], dtype=np.float32), k)

    top_k_indices_list = top_k_indices[0].tolist()
    top_k_distances_list = top_k_distances[0].tolist()

    return top_k_indices_list, top_k_distances_list


Создание индекса faiss может занять некоторое время, это нормально. В каком-то смысле мы обмениваем вычисления во время построения индекса на время выполнения запроса в дальнейшем.

In [ ]:
embedding_dimension = finetuned_model.vector_size
hnsw_index = build_faiss_hnsw_index(embedding_dimension)

populate_faiss_index(hnsw_index, finetuned_model, "quora_processed.txt")


Заметим, что мы построили индекс FAISS вокруг Евклидова расстояния, а не косинусного сходства, если быть до конца честными:

$$
cosine \space similarity = 1 - \frac{euclidian \space distance^2}{2}
$$

Обратите внимание, что полученные результаты могут не полностью повторять поиск на основе numpy, так как индекс HNSW построен на приближенных вычислениях.

In [ ]:
query_text = "How can I find inner peace?"

top_k_indices, top_k_similarities = fetch_and_display_closest_match(query_function=search_faiss_index,
                                                                    query=query_text, model=finetuned_model,
                                                                    embeddings_storage=hnsw_index,
                                                                    k=10)

Запрос: How can I find inner peace?

Лучшие совпадения:
1. How can I find passion? (Сходство: -0.0128)
2. How can I find inspiration? (Сходство: -0.0140)
3. How can I find a career mentor? (Сходство: -0.0144)
4. How can I find a successful mentor? (Сходство: -0.0149)
5. How can I find a programmer mentor? (Сходство: -0.0157)
6. How can I find someone's MSN profile? (Сходство: -0.0164)
7. How can I truly find happiness? (Сходство: -0.0178)
8. How can I find a startup mentor? (Сходство: -0.0178)
9. How can I prevent business failure? (Сходство: -0.0179)
10. How can I find a source code? (Сходство: -0.0181)


In [58]:
import time

iterations = 100

start_time = time.time()

for _ in range(iterations):
    _, _ = search_faiss_index(hnsw_index, query_text, finetuned_model, k=10)

time_elapsed = time.time() - start_time
average_time = time_elapsed / iterations

print(f'Прошло времени: {time_elapsed:.4f} секунд.')
print(f'Среднее время: {average_time:.4f} секунд.')

Прошло времени: 0.0137 секунд.
Среднее время: 0.0001 секунд.


Обратите внимание, что мы добились ускорения на два порядка!

**Поздравляем, вы справились с заданием!** Но куда смотреть дальше если хочется большего?

Что касается эмбеддингов, вы можете захотеть улучшить то, как получается эмбеддинг отдельного слова, или ввести индивидуальное взвешивание эмбеддингов, например, с помощью TF-IDF. Однако, конечно, наилуших результатов можно добиться используя способы представления произвольного текста, основанные на архитектуре трансформера (см. например [Sentence Transformers](https://huggingface.co/models?library=sentence-transformers&sort=trending).

Что касается поисковой системы, кэширование — отличный вариант для избавления от избыточных запросов. Например, взгляните на [Redis](https://redis.io/learn/howtos/solutions/microservices/caching).